In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. 데이터 전처리 및 파생변수 생성
# ============================================

def extract_interval_mean(interval_str):
    """구간 문자열에서 평균값 추출"""
    if pd.isna(interval_str):
        return np.nan
    
    interval_str = str(interval_str).strip()
    
    try:
        # 이미 숫자인 경우
        return float(interval_str)
    except:
        pass
    
    try:
        # '5_75-90%' 형태 처리 (구간 순위_퍼센트 범위)
        if '_' in interval_str and '%' in interval_str:
            # '_' 뒤의 부분만 추출
            parts = interval_str.split('_')[1]
            # '75-90%' 또는 '90%초과' 등 처리
            if '-' in parts:
                # '75-90%' -> 75와 90의 평균
                numbers = parts.replace('%', '').split('-')
                return np.mean([float(x) for x in numbers])
            elif '초과' in parts:
                # '90%초과' -> 95 (상위 5%)
                num = float(parts.replace('%초과', '').replace('(하위', '').split('%')[0])
                return (num + 100) / 2
            elif '미만' in parts:
                # '10%미만' -> 5
                num = float(parts.replace('%미만', ''))
                return num / 2
            else:
                # 단일 숫자
                return float(parts.replace('%', ''))
        
        # '1000-5000' 형태 처리
        elif '-' in interval_str and '%' not in interval_str:
            numbers = [float(x.strip().replace(',', '')) for x in interval_str.split('-')]
            return np.mean(numbers)
        
        # '1000이상' 형태 처리
        elif '이상' in interval_str:
            num = float(interval_str.replace('이상', '').replace(',', '').strip())
            return num
        
        # '1000미만' 형태 처리
        elif '미만' in interval_str:
            num = float(interval_str.replace('미만', '').replace(',', '').strip())
            return num * 0.5
        
        else:
            return np.nan
    except Exception as e:
        return np.nan


def calculate_customer_diversity(row):
    """고객 분포 다양성 지수 (섀넌 엔트로피)"""
    
    customer_cols = [
        '남성 20대이하 고객 비중', '남성 30대 고객 비중', '남성 40대 고객 비중',
        '남성 50대 고객 비중', '남성 60대이상 고객 비중',
        '여성 20대이하 고객 비중', '여성 30대 고객 비중', '여성 40대 고객 비중',
        '여성 50대 고객 비중', '여성 60대이상 고객 비중'
    ]
    
    proportions = []
    for col in customer_cols:
        if col in row.index and pd.notna(row[col]):
            val = row[col]
            if val > 0:
                proportions.append(val)
    
    if not proportions or sum(proportions) == 0:
        return 0
    
    # 정규화
    proportions = [p / sum(proportions) for p in proportions]
    
    # 섀넌 엔트로피 계산
    entropy = -sum(p * np.log(p + 1e-10) for p in proportions if p > 0)
    return entropy


def calculate_monthly_changes(df):
    """전월 대비 변화율 계산"""
    
    df = df.sort_values(['가맹점구분번호', '기준년월'])
    
    # 취소율 변화율 (수치 사용)
    if '취소율 구간_수치' in df.columns:
        df['취소율_전월대비변화율'] = df.groupby('가맹점구분번호')['취소율 구간_수치'].pct_change()
    
    # 매출건수 변화율
    if '매출건수 구간_수치' in df.columns:
        df['매출건수_변화율'] = df.groupby('가맹점구분번호')['매출건수 구간_수치'].pct_change()
    
    # 매출금액 변화율
    if '매출금액 구간_수치' in df.columns:
        df['매출금액_변화율'] = df.groupby('가맹점구분번호')['매출금액 구간_수치'].pct_change()
    
    # 객단가 변화율
    if '객단가 구간_수치' in df.columns:
        df['객단가_변화율'] = df.groupby('가맹점구분번호')['객단가 구간_수치'].pct_change()
    
    # 신규고객 변화율
    if '신규 고객 비중' in df.columns:
        df['신규고객_변화율'] = df.groupby('가맹점구분번호')['신규 고객 비중'].pct_change()
    
    return df


def preprocess_data(df):
    """데이터 전처리 및 파생변수 생성"""
    
    print("전처리 시작...")
    
    # 날짜 변환
    if df['기준년월'].dtype == 'object' or df['기준년월'].dtype == 'int64':
        df['기준년월'] = pd.to_datetime(df['기준년월'].astype(str), format='%Y%m', errors='coerce')
    
    df = df.sort_values(['가맹점구분번호', '기준년월'])
    
    print("1. 구간 변수 변환 중...")
    # 구간 변수들을 숫자로 변환
    interval_cols = ['매출금액 구간', '매출건수 구간', '유니크 고객 수 구간', 
                     '객단가 구간']
    
    for col in interval_cols:
        if col in df.columns:
            print(f"   - {col} 변환 중... (샘플: {df[col].iloc[0]})")
            df[col + '_수치'] = df[col].apply(extract_interval_mean)
            non_null_count = df[col + '_수치'].notna().sum()
            print(f"     변환 완료: {non_null_count}/{len(df)} 값")
    
    # 취소율 구간 처리
    if '취소율 구간' in df.columns:
        print(f"   - 취소율 구간 변환 중... (샘플: {df['취소율 구간'].iloc[0]})")
        # '1_상위1구간' 형태 처리
        def parse_cancellation_rate(x):
            if pd.isna(x):
                return np.nan
            x_str = str(x).strip()
            # '1_상위1구간' -> 1 추출
            if '_' in x_str:
                try:
                    return float(x_str.split('_')[0])
                except:
                    return np.nan
            try:
                return float(x_str)
            except:
                return np.nan
        
        df['취소율 구간_수치'] = df['취소율 구간'].apply(parse_cancellation_rate)
        print(f"     변환 완료: {df['취소율 구간_수치'].notna().sum()}/{len(df)} 값")
    
    print("2. 전월 대비 변화율 계산 중...")
    # 전월 대비 변화율 계산
    df = calculate_monthly_changes(df)
    
    # 변화율 계산 결과 확인
    change_cols = ['취소율_전월대비변화율', '매출건수_변화율', '매출금액_변화율', '객단가_변화율']
    print("   변화율 계산 결과:")
    for col in change_cols:
        if col in df.columns:
            non_null = df[col].notna().sum()
            print(f"   - {col}: {non_null}/{len(df)} 값")
    
    print("3. 고객 분포 다양성 지수 계산 중...")
    # 고객 분포 다양성 지수 계산
    df['고객분포_다양성지수'] = df.apply(calculate_customer_diversity, axis=1)
    
    # 무한대 값 제거
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    
    print("전처리 완료!")
    return df


# ============================================
# 2. 모델 1: 취소율 급증 예측 모델
# ============================================

def create_cancellation_spike_target(df, threshold_percentile=80):
    """취소율 급증 여부 타겟 변수 생성"""
    
    # 취소율 변화율이 있는 데이터만 사용
    df_valid = df[df['취소율_전월대비변화율'].notna()].copy()
    
    if len(df_valid) == 0:
        print("경고: 취소율 변화율 데이터가 없습니다.")
        return df
    
    # 취소율 변화율 상위 threshold_percentile% 를 급증으로 정의
    threshold = df_valid['취소율_전월대비변화율'].quantile(threshold_percentile / 100)
    df['취소율_급증여부'] = 0
    df.loc[df['취소율_전월대비변화율'] > threshold, '취소율_급증여부'] = 1
    
    return df


def build_cancellation_model(df):
    """취소율 급증 예측 모델 구축"""
    
    print("=" * 60)
    print("모델 1: 취소율 급증 예측 모델")
    print("=" * 60)
    
    # 타겟 변수 생성
    df = create_cancellation_spike_target(df)
    
    # 독립변수 선택
    feature_cols = [
        '배달매출금액 비율',
        '동일 업종 매출금액 비율',
        '신규 고객 비중',
        '재방문 고객 비중',
        '취소율_전월대비변화율',
        '매출건수_변화율',
        '객단가_변화율'
    ]
    
    # 신규고객 변화율이 있으면 추가
    if '신규고객_변화율' in df.columns:
        feature_cols.append('신규고객_변화율')
    
    # 실제 존재하는 컬럼만 선택
    available_features = [col for col in feature_cols if col in df.columns]
    print(f"\n사용 가능한 독립변수: {available_features}")
    
    # 결측치 제거 전 데이터 확인
    print(f"\n결측치 확인:")
    for col in available_features:
        if col in df.columns:
            null_count = df[col].isna().sum()
            null_pct = null_count / len(df) * 100
            print(f"  {col}: {null_count} ({null_pct:.1f}%)")
    
    # 타겟 변수 확인
    if '취소율_급증여부' in df.columns:
        null_count = df['취소율_급증여부'].isna().sum()
        null_pct = null_count / len(df) * 100
        print(f"  취소율_급증여부: {null_count} ({null_pct:.1f}%)")
    else:
        print("  취소율_급증여부: 변수가 생성되지 않았습니다.")
        print("\n오류: 타겟 변수가 생성되지 않았습니다. 취소율 데이터를 확인해주세요.")
        return None, None, None, available_features
    
    # 결측치 제거
    model_df = df[available_features + ['취소율_급증여부']].dropna()
    
    if len(model_df) == 0:
        print("\n오류: 결측치 제거 후 데이터가 없습니다.")
        print("데이터 확인이 필요합니다.")
        return None, None, None, available_features
    
    X = model_df[available_features]
    y = model_df['취소율_급증여부']
    
    print(f"\n샘플 수: {len(model_df)}")
    print(f"취소율 급증 비율: {y.mean():.2%}")
    
    # 클래스가 하나만 있는지 확인
    if y.nunique() < 2:
        print("\n오류: 타겟 변수에 클래스가 하나만 있습니다.")
        return None, None, None, available_features
    
    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 표준화
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 로지스틱 회귀 모델
    log_model = LogisticRegression(random_state=42, max_iter=1000)
    log_model.fit(X_train_scaled, y_train)
    
    # 랜덤 포레스트 모델
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    rf_model.fit(X_train_scaled, y_train)
    
    # 예측 및 평가
    print("\n[로지스틱 회귀 결과]")
    evaluate_model(log_model, X_test_scaled, y_test)
    
    print("\n[랜덤 포레스트 결과]")
    evaluate_model(rf_model, X_test_scaled, y_test)
    
    # 변수 중요도 (랜덤 포레스트)
    print("\n[변수 중요도 - Random Forest]")
    feature_importance = pd.DataFrame({
        '변수': available_features,
        '중요도': rf_model.feature_importances_
    }).sort_values('중요도', ascending=False)
    print(feature_importance.to_string(index=False))
    
    # 로지스틱 회귀 계수
    print("\n[로지스틱 회귀 계수]")
    coef_df = pd.DataFrame({
        '변수': available_features,
        '계수': log_model.coef_[0]
    }).sort_values('계수', ascending=False)
    print(coef_df.to_string(index=False))
    
    return log_model, rf_model, scaler, available_features


# ============================================
# 3. 모델 2: 매출 변동성 예측 모델
# ============================================

def create_volatility_target(df, window=3, threshold_percentile=80):
    """매출 변동성 타겟 변수 생성 (최근 3개월 표준편차 상위 20%)"""
    
    df = df.sort_values(['가맹점구분번호', '기준년월'])
    
    # 최근 3개월 매출금액 표준편차 계산
    if '매출금액 구간_수치' in df.columns:
        df['매출_3개월_표준편차'] = df.groupby('가맹점구분번호')['매출금액 구간_수치'].rolling(
            window=window, min_periods=2
        ).std().reset_index(level=0, drop=True)
        
        # 상위 threshold_percentile% 를 고변동성으로 정의
        df_valid = df[df['매출_3개월_표준편차'].notna()].copy()
        
        if len(df_valid) > 0:
            threshold = df_valid['매출_3개월_표준편차'].quantile(threshold_percentile / 100)
            df['매출_고변동성여부'] = 0
            df.loc[df['매출_3개월_표준편차'] >= threshold, '매출_고변동성여부'] = 1
    
    return df


def build_volatility_model(df):
    """매출 변동성 예측 모델 구축"""
    
    print("\n\n" + "=" * 60)
    print("모델 2: 매출 변동성 예측 모델")
    print("=" * 60)
    
    # 타겟 변수 생성
    df = create_volatility_target(df)
    
    if '매출_고변동성여부' not in df.columns:
        print("\n오류: 타겟 변수 생성 실패")
        return None, None, None, []
    
    # 독립변수 선택
    feature_cols = [
        '매출금액_변화율',
        '매출건수_변화율',
        '동일 상권 내 매출 순위 비율',
        '동일 업종 내 해지 가맹점 비중',
        '고객분포_다양성지수',
        '동일 업종 매출금액 비율',
        '재방문 고객 비중'
    ]
    
    # 실제 존재하는 컬럼만 선택
    available_features = [col for col in feature_cols if col in df.columns]
    print(f"\n사용 가능한 독립변수: {available_features}")
    
    # 결측치 제거 전 데이터 확인
    print(f"\n결측치 확인:")
    for col in available_features:
        if col in df.columns:
            null_count = df[col].isna().sum()
            null_pct = null_count / len(df) * 100
            print(f"  {col}: {null_count} ({null_pct:.1f}%)")
    
    # 타겟 변수 확인
    if '매출_고변동성여부' in df.columns:
        null_count = df['매출_고변동성여부'].isna().sum()
        null_pct = null_count / len(df) * 100
        print(f"  매출_고변동성여부: {null_count} ({null_pct:.1f}%)")
    else:
        print("  매출_고변동성여부: 변수가 생성되지 않았습니다.")
    
    # 결측치 제거
    model_df = df[available_features + ['매출_고변동성여부']].dropna()
    
    if len(model_df) == 0:
        print("\n오류: 결측치 제거 후 데이터가 없습니다.")
        return None, None, None, available_features
    
    X = model_df[available_features]
    y = model_df['매출_고변동성여부']
    
    print(f"\n샘플 수: {len(model_df)}")
    print(f"고변동성 비율: {y.mean():.2%}")
    
    # 클래스가 하나만 있는지 확인
    if y.nunique() < 2:
        print("\n오류: 타겟 변수에 클래스가 하나만 있습니다.")
        return None, None, None, available_features
    
    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # 표준화
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 로지스틱 회귀 모델
    log_model = LogisticRegression(random_state=42, max_iter=1000)
    log_model.fit(X_train_scaled, y_train)
    
    # 랜덤 포레스트 모델
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    rf_model.fit(X_train_scaled, y_train)
    
    # 예측 및 평가
    print("\n[로지스틱 회귀 결과]")
    evaluate_model(log_model, X_test_scaled, y_test)
    
    print("\n[랜덤 포레스트 결과]")
    evaluate_model(rf_model, X_test_scaled, y_test)
    
    # 변수 중요도
    print("\n[변수 중요도 - Random Forest]")
    feature_importance = pd.DataFrame({
        '변수': available_features,
        '중요도': rf_model.feature_importances_
    }).sort_values('중요도', ascending=False)
    print(feature_importance.to_string(index=False))
    
    # 로지스틱 회귀 계수
    print("\n[로지스틱 회귀 계수]")
    coef_df = pd.DataFrame({
        '변수': available_features,
        '계수': log_model.coef_[0]
    }).sort_values('계수', ascending=False)
    print(coef_df.to_string(index=False))
    
    return log_model, rf_model, scaler, available_features


def evaluate_model(model, X_test, y_test):
    """모델 평가"""
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n정확도: {model.score(X_test, y_test):.4f}")
    print(f"AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
    
    print("\n분류 리포트:")
    print(classification_report(y_test, y_pred, target_names=['정상', '위험']))
    
    print("\n혼동 행렬:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(f"\nTN: {cm[0,0]}, FP: {cm[0,1]}")
    print(f"FN: {cm[1,0]}, TP: {cm[1,1]}")


# ============================================
# 4. 메인 실행 함수
# ============================================

def main(df):
    """전체 분석 파이프라인 실행"""
    
    print("데이터 전처리 시작...")
    print(f"원본 데이터: {len(df)} 행, {len(df.columns)} 컬럼")
    print(f"가맹점 수: {df['가맹점구분번호'].nunique()}")
    
    df_processed = preprocess_data(df.copy())
    
    print(f"\n전처리 완료. 총 {len(df_processed)} 행")
    
    # 모델 1: 취소율 급증 예측
    cancel_log, cancel_rf, cancel_scaler, cancel_features = build_cancellation_model(
        df_processed.copy()
    )
    
    # 모델 2: 매출 변동성 예측
    vol_log, vol_rf, vol_scaler, vol_features = build_volatility_model(
        df_processed.copy()
    )
    
    print("\n\n" + "=" * 60)
    print("분석 완료!")
    print("=" * 60)
    
    return {
        'processed_data': df_processed,
        'cancellation_models': {
            'logistic': cancel_log,
            'random_forest': cancel_rf,
            'scaler': cancel_scaler,
            'features': cancel_features
        },
        'volatility_models': {
            'logistic': vol_log,
            'random_forest': vol_rf,
            'scaler': vol_scaler,
            'features': vol_features
        }
    }


# ============================================
# 실행 예시
# ============================================

# 데이터 로드
df = pd.read_csv('merged_df.csv')

# 분석 실행
results = main(df)

# 결과 활용
if results['cancellation_models']['random_forest'] is not None:
    processed_data = results['processed_data']
    print("\n모델 학습 완료!")
else:
    print("\n모델 학습 실패. 데이터를 확인해주세요.")

데이터 전처리 시작...
원본 데이터: 1960290 행, 39 컬럼
가맹점 수: 4185
전처리 시작...
1. 구간 변수 변환 중...
   - 매출금액 구간 변환 중... (샘플: 6_90%초과(하위 10% 이하))
     변환 완료: 1575864/1960290 값
   - 매출건수 구간 변환 중... (샘플: 5_75-90%)
     변환 완료: 1614267/1960290 값
   - 유니크 고객 수 구간 변환 중... (샘플: 5_75-90%)
     변환 완료: 1613085/1960290 값
   - 객단가 구간 변환 중... (샘플: 4_50-75%)
     변환 완료: 1581470/1960290 값
   - 취소율 구간 변환 중... (샘플: 1_상위1구간)
     변환 완료: 1812539/1960290 값
2. 전월 대비 변화율 계산 중...
   변화율 계산 결과:
   - 취소율_전월대비변화율: 1923243/1960290 값
   - 매출건수_변화율: 1796021/1960290 값
   - 매출금액_변화율: 1805072/1960290 값
   - 객단가_변화율: 1782930/1960290 값
3. 고객 분포 다양성 지수 계산 중...
전처리 완료!

전처리 완료. 총 1960290 행
모델 1: 취소율 급증 예측 모델

사용 가능한 독립변수: ['배달매출금액 비율', '동일 업종 매출금액 비율', '신규 고객 비중', '재방문 고객 비중', '취소율_전월대비변화율', '매출건수_변화율', '객단가_변화율', '신규고객_변화율']

결측치 확인:
  배달매출금액 비율: 0 (0.0%)
  동일 업종 매출금액 비율: 0 (0.0%)
  신규 고객 비중: 0 (0.0%)
  재방문 고객 비중: 0 (0.0%)
  취소율_전월대비변화율: 37047 (1.9%)
  매출건수_변화율: 164269 (8.4%)
  객단가_변화율: 177360 (9.0%)
  신규고객_변화율: 160353 (8.2%)
  취소율_급증여부: 0 (